# 🚩 Sprint 1 · Checkpoint 05 — The Showdown (Capstone)

**Companion ao Boyd (Cap 9.5 — Newton's Method, Cap 9.7 — Implementation) + Weinberger Lec 12.**

Capstone de Sprint 1. Reúnes todos os otimizadores que codaste nos CPs 01-04 e pões-nos a competir num teste canónico — a função de **Rosenbrock**. Aprendes três ferramentas profissionais:

1. **Numerical gradient check** — como confirmar que a tua derivada analítica está certa.
2. **Backtracking line search com Armijo** — como escolher o tamanho do passo automaticamente.
3. **Análise de paisagem (eigenvalues da Hessiana)** — distinguir mínimo, máximo, e ponto-de-sela.

E adicionas dois otimizadores extra: **Nadam** (Nesterov + Adam) e um **scheduler** de learning rate.

**Como trabalhar:**
- Cada `# TODO` é uma micro-task. Substitui `...` ou completa o esqueleto.
- A célula corre uma `assert` — se falhar, lê a mensagem e corrige.
- Trabalha em `solutions/local.ipynb`.

## ⚙️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

def check(cond, msg='falhou'):
    assert cond, f'❌ {msg}'
    print('✅', msg)

# Rosenbrock — o nosso campo de testes
def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

def rosenbrock_grad_analytical(x):
    """Derivada à mão — TASK 1.1 vai verificar se está certa."""
    df_dx = -2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2)
    df_dy = 200 * (x[1] - x[0]**2)
    return np.array([df_dx, df_dy])

def rosenbrock_hess(x):
    """Hessiana analítica (também 2x2)."""
    h_xx = 2 - 400 * (x[1] - 3 * x[0]**2)
    h_xy = -400 * x[0]
    h_yy = 200.0
    return np.array([[h_xx, h_xy], [h_xy, h_yy]])

## §1 — Gradient check: nunca acredites num gradiente que não testaste

Em ML real vais derivar gradientes complicados à mão (ou em PyTorch/JAX). 80% dos bugs em redes neurais não-convergentes vêm de **gradientes errados**. A defesa: comparar com o numérico em pontos aleatórios.

**Erro relativo** é a métrica certa (não o absoluto):

$$ \text{rel\_err} = \frac{\|g_{\text{analy}} - g_{\text{num}}\|}{\max(\|g_{\text{analy}}\|, \|g_{\text{num}}\|, 10^{-9})} $$

**Threshold:** `< 1e-5` é "claramente certo", `1e-5 a 1e-4` é "provavelmente certo (h pequeno demais ou função muito não-suave)", `> 1e-4` é "tens um bug".

### Task 1.1 — Implementa `gradient_check`

In [ ]:
def numerical_grad(f, x, h=1e-5):
    x = np.asarray(x, dtype=float)
    g = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy(); x_plus[i] += h
        x_minus = x.copy(); x_minus[i] -= h
        g[i] = (f(x_plus) - f(x_minus)) / (2 * h)
    return g

def gradient_check(f, grad_fn, x, h=1e-5):
    """Devolve o erro relativo entre grad analítico e numérico em x."""
    g_analy = grad_fn(x)
    g_num = numerical_grad(f, x, h=h)
    # TODO: numerador = norm(g_analy - g_num)
    # TODO: denom = max(norm(g_analy), norm(g_num), 1e-9)
    # TODO: devolve numerador / denom
    return ...

# Teste 1: Rosenbrock — grad analítico está certo
for _ in range(5):
    x = rng.uniform(-2, 2, size=2)
    err = gradient_check(rosenbrock, rosenbrock_grad_analytical, x)
    print(f'x={x}, rel_err={err:.2e}')
    check(err < 1e-5, f'gradiente analítico passa o check em x={x}, err={err:.2e}')

# Teste 2: gradiente DELIBERADAMENTE errado deve ser detectado
def buggy_grad(x):
    df_dx = -2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2)
    df_dy = 100 * (x[1] - x[0]**2)  # <-- BUG: 100 em vez de 200
    return np.array([df_dx, df_dy])

err_buggy = gradient_check(rosenbrock, buggy_grad, np.array([0.5, 0.3]))
check(err_buggy > 1e-3, f'check deve detetar bug do factor 2, err={err_buggy:.2e}')

## §2 — Backtracking line search (Armijo)

No CP04 fizeste line search por halving naïve: `f(x + αp) < f(x)`. Aceitável, mas teoricamente fraco — pode aceitar passos minúsculos que mal melhoram.

A **condição de Armijo** garante que o passo melhora *suficientemente*:

$$ f(x + \alpha p) \le f(x) + c\, \alpha\, g^T p $$

Onde `c ∈ (0, 1)` (tipicamente `c=1e-4`). Em palavras: a redução real tem que ser pelo menos uma fracção da redução prevista pelo gradiente. Se não, divide $\alpha$ por 2 e repete.

### Task 2.1 — Implementa `backtracking_armijo`

In [ ]:
def backtracking_armijo(f, x, p, g, alpha_init=1.0, c=1e-4, rho=0.5, max_iter=50):
    """Backtracking com Armijo. Devolve o alpha que satisfaz f(x+αp) ≤ f(x) + c α gᵀp."""
    f_curr = f(x)
    descent = np.dot(g, p)  # deve ser < 0 se p é direção de descida
    alpha = alpha_init
    for _ in range(max_iter):
        # TODO: se f(x + alpha*p) <= f_curr + c * alpha * descent, retorna alpha
        # TODO: caso contrário, alpha = rho * alpha
        ...
    return alpha  # devolvemos o melhor que encontrámos

# Teste: em Rosenbrock, a partir de (-1.2, 1), com direção -g
x = np.array([-1.2, 1.0])
g = rosenbrock_grad_analytical(x)
p = -g  # steepest descent
alpha = backtracking_armijo(rosenbrock, x, p, g)
print(f'alpha encontrado: {alpha:.6f}')
print(f'f(x) = {rosenbrock(x):.4f}, f(x + α p) = {rosenbrock(x + alpha * p):.4f}')

check(alpha > 0, f'alpha deve ser positivo, obteve {alpha}')
check(rosenbrock(x + alpha * p) < rosenbrock(x),
      f'f deve diminuir; antes={rosenbrock(x):.4f}, depois={rosenbrock(x + alpha * p):.4f}')

# Condição de Armijo cumprida
check(rosenbrock(x + alpha * p) <= rosenbrock(x) + 1e-4 * alpha * np.dot(g, p) + 1e-9,
      'condição de Armijo deve estar satisfeita')

## §3 — Nadam: Nesterov + Adam

Adam usa momentum normal. **Nadam** (Dozat, 2016) substitui-o por momentum tipo Nesterov:

Em Adam o update é $\hat{m}/(\sqrt{\hat{v}}+\epsilon)$.
Em Nadam, $\hat{m}$ é substituído por uma versão "lookahead":

$$ m\_lookahead = \beta_1 \hat{m}_{t} + \frac{(1 - \beta_1)\, g_t}{1 - \beta_1^{t+1}} $$

Update: $x_{t+1} = x_t - \eta\, m\_lookahead / (\sqrt{\hat{v}_t} + \epsilon)$.

Em prática, Nadam tende a convergir um pouco mais rápido que Adam em early iterations.

### Task 3.1 — Implementa Nadam

In [ ]:
def nadam(x0, grad_fn, lr=0.001, beta1=0.9, beta2=0.999, n_iter=200, eps=1e-8):
    x = np.asarray(x0, dtype=float).copy()
    m = np.zeros_like(x)
    v = np.zeros_like(x)
    history = [x.copy()]
    for t in range(n_iter):
        g = grad_fn(x)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g * g
        m_hat = m / (1 - beta1 ** (t + 1))
        v_hat = v / (1 - beta2 ** (t + 1))
        # TODO: m_lookahead = beta1 * m_hat + (1 - beta1) * g / (1 - beta1 ** (t + 1))
        # TODO: x = x - lr * m_lookahead / (np.sqrt(v_hat) + eps)
        ...
        history.append(x.copy())
    return np.array(history)

# Teste em Rosenbrock — Nadam deve atingir (1, 1) em 2000 iter como Adam
hist_nadam = nadam(np.array([-1.2, 1.0]), rosenbrock_grad_analytical, lr=0.01, n_iter=2000)
final_nadam = hist_nadam[-1]
print(f'Nadam final: {final_nadam}, f={rosenbrock(final_nadam):.6f}')
check(rosenbrock(final_nadam) < 0.05,
      f'Nadam deve aproximar (1,1) em 2000 iter, obteve f={rosenbrock(final_nadam):.6f}')

### Task 3.2 — Cosine learning rate scheduler

Em treino de redes modernas, baixar a `lr` ao longo do treino é quase sempre bom. O **cosine scheduler** (Loshchilov & Hutter 2017) varia `lr` entre `lr_max` e `lr_min` numa curva cosseno:

$$ \eta_t = \eta_{\min} + \tfrac{1}{2}(\eta_{\max} - \eta_{\min})\Big(1 + \cos\Big(\pi\, \frac{t}{T}\Big)\Big) $$

Onde $T$ é o total de iterações.

In [ ]:
def cosine_lr(t, T, lr_max, lr_min=0.0):
    """Cosine decay: começa em lr_max, termina em lr_min após T passos."""
    # TODO: usa np.cos para devolver lr_min + 0.5 * (lr_max - lr_min) * (1 + cos(π t / T))
    return ...

# Sanity
T = 100
check(abs(cosine_lr(0, T, 1.0, 0.0) - 1.0) < 1e-9, 'em t=0, lr=lr_max')
check(abs(cosine_lr(T, T, 1.0, 0.0) - 0.0) < 1e-9, 'em t=T, lr=lr_min')
check(abs(cosine_lr(T // 2, T, 1.0, 0.0) - 0.5) < 1e-3, 'em t=T/2, lr ≈ (max+min)/2')

# Plot da curva
ts = np.arange(T + 1)
lrs = np.array([cosine_lr(t, T, 0.1, 0.001) for t in ts])
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(ts, lrs)
ax.set_xlabel('iteração'); ax.set_ylabel('lr'); ax.set_title('Cosine LR decay')
plt.show()

## §4 — The Showdown: 7 otimizadores na Rosenbrock

Tens agora um arsenal: **GD, Momentum, NAG, AdaGrad, RMSProp, Adam, Nadam, Newton com line search**. Vamos pô-los todos a competir.

Esta secção fornece **implementações de referência** (são as mesmas que codaste em CPs 01-04 — copia as tuas se preferires). O foco aqui é a *análise comparativa*.

### Task 4.1 — Setup do showdown

In [ ]:
# Implementações de referência (já codadas em CPs anteriores)
def gd(x0, grad_fn, lr, n_iter):
    x = np.asarray(x0, dtype=float).copy()
    history = [x.copy()]
    for _ in range(n_iter):
        x = x - lr * grad_fn(x)
        history.append(x.copy())
    return np.array(history)

def gd_momentum(x0, grad_fn, lr, beta, n_iter):
    x = np.asarray(x0, dtype=float).copy()
    v = np.zeros_like(x); history = [x.copy()]
    for _ in range(n_iter):
        v = beta * v + grad_fn(x)
        x = x - lr * v
        history.append(x.copy())
    return np.array(history)

def gd_nesterov(x0, grad_fn, lr, beta, n_iter):
    x = np.asarray(x0, dtype=float).copy()
    v = np.zeros_like(x); history = [x.copy()]
    for _ in range(n_iter):
        lookahead = x - lr * beta * v
        g = grad_fn(lookahead)
        v = beta * v + g
        x = x - lr * v
        history.append(x.copy())
    return np.array(history)

def adagrad(x0, grad_fn, lr, n_iter, eps=1e-8):
    x = np.asarray(x0, dtype=float).copy()
    G = np.zeros_like(x); history = [x.copy()]
    for _ in range(n_iter):
        g = grad_fn(x); G = G + g * g
        x = x - lr * g / (np.sqrt(G) + eps)
        history.append(x.copy())
    return np.array(history)

def rmsprop(x0, grad_fn, lr, rho=0.9, n_iter=200, eps=1e-8):
    x = np.asarray(x0, dtype=float).copy()
    s = np.zeros_like(x); history = [x.copy()]
    for _ in range(n_iter):
        g = grad_fn(x); s = rho * s + (1 - rho) * g * g
        x = x - lr * g / (np.sqrt(s) + eps)
        history.append(x.copy())
    return np.array(history)

def adam(x0, grad_fn, lr=0.001, beta1=0.9, beta2=0.999, n_iter=200, eps=1e-8):
    x = np.asarray(x0, dtype=float).copy()
    m = np.zeros_like(x); v = np.zeros_like(x); history = [x.copy()]
    for t in range(n_iter):
        g = grad_fn(x)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g * g
        m_hat = m / (1 - beta1 ** (t + 1))
        v_hat = v / (1 - beta2 ** (t + 1))
        x = x - lr * m_hat / (np.sqrt(v_hat) + eps)
        history.append(x.copy())
    return np.array(history)

def newton_armijo(f, grad_fn, hess_fn, x0, n_iter):
    x = np.asarray(x0, dtype=float).copy()
    history = [x.copy()]
    for _ in range(n_iter):
        g = grad_fn(x); H = hess_fn(x)
        try: p = -np.linalg.solve(H, g)
        except np.linalg.LinAlgError: p = -g
        if np.dot(p, g) >= 0: p = -g
        alpha = backtracking_armijo(f, x, p, g, alpha_init=1.0)
        x = x + alpha * p
        history.append(x.copy())
    return np.array(history)

print('🔧 7 otimizadores prontos.')

### Task 4.2 — Corre o showdown e compara

Mesma config: ponto inicial `(-1.2, 1)`, 2000 iterações, lr afinada por método.

**Predição (escreve antes de correr e confirma depois):**

- Newton com line search: melhor (já viste no CP04).
- Adam, Nadam, RMSProp: bons, com Nadam um pouco à frente.
- Momentum, NAG: razoáveis com lr afinada.
- GD, AdaGrad: medíocre. AdaGrad morre.
- GD puro: pior.

In [ ]:
x0 = np.array([-1.2, 1.0])
N = 2000

# Corre cada método com hyperparams afinados (típicos)
runs = {
    'GD':       gd(x0, rosenbrock_grad_analytical, lr=0.001, n_iter=N),
    'Momentum': gd_momentum(x0, rosenbrock_grad_analytical, lr=0.001, beta=0.9, n_iter=N),
    'Nesterov': gd_nesterov(x0, rosenbrock_grad_analytical, lr=0.001, beta=0.9, n_iter=N),
    'AdaGrad':  adagrad(x0, rosenbrock_grad_analytical, lr=0.5, n_iter=N),
    'RMSProp':  rmsprop(x0, rosenbrock_grad_analytical, lr=0.05, n_iter=N),
    'Adam':     adam(x0, rosenbrock_grad_analytical, lr=0.01, n_iter=N),
    'Nadam':    nadam(x0, rosenbrock_grad_analytical, lr=0.01, n_iter=N),
    'Newton':   newton_armijo(rosenbrock, rosenbrock_grad_analytical, rosenbrock_hess, x0, n_iter=200),  # menos iter — converge muito rápido
}

# Tabela final
print(f'{"Method":<12} {"f_final":>14} {"||x-(1,1)||":>14}')
print('-' * 42)
for name, hist in runs.items():
    f_final = rosenbrock(hist[-1])
    dist = np.linalg.norm(hist[-1] - np.array([1.0, 1.0]))
    print(f'{name:<12} {f_final:>14.6e} {dist:>14.6e}')

# Newton tem que ser o melhor (de longe)
f_newton = rosenbrock(runs['Newton'][-1])
f_others = [rosenbrock(h[-1]) for name, h in runs.items() if name != 'Newton']
check(f_newton < min(f_others),
      f'Newton tem que ter o menor f_final, Newton={f_newton:.2e}, melhor outro={min(f_others):.2e}')

# Curvas de convergência (log de f vs iteração)
fig, ax = plt.subplots(figsize=(9, 5))
for name, hist in runs.items():
    losses = np.array([rosenbrock(p) for p in hist])
    ax.semilogy(losses, label=name)
ax.set_xlabel('iteração'); ax.set_ylabel('Rosenbrock(x) [log]')
ax.set_title('7 otimizadores em Rosenbrock — convergência')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.show()

## §5 — Análise da paisagem: eigenvalues da Hessiana

A Hessiana num ponto `x` diz-te a **curvatura local**. Os valores próprios distinguem três tipos de pontos críticos (onde $\nabla f = 0$):

- **Mínimo local:** todos os eigenvalues > 0 (Hessiana definida positiva).
- **Máximo local:** todos < 0 (definida negativa).
- **Ponto de sela:** alguns > 0, outros < 0 (indefinida).

Em deep learning, a maior parte dos pontos críticos são pontos de sela (não mínimos). Isto explica porque optimizadores adaptativos brilham — escapam-se de selas mais rapidamente.

### Task 5.1 — Classifica pontos via eigenvalues

Considera $f(x, y) = x^2 - y^2$. Tem um único ponto crítico em $(0, 0)$ — onde $\nabla f = 0$. Mas é um **ponto de sela**: ascende em $x$, descende em $y$. A Hessiana é $\text{diag}(2, -2)$ — eigenvalues mistos.

In [ ]:
def saddle_f(x):
    return x[0]**2 - x[1]**2

def saddle_grad(x):
    return np.array([2 * x[0], -2 * x[1]])

def saddle_hess(x):
    return np.array([[2.0, 0.0], [0.0, -2.0]])

def classify_critical_point(hess):
    """Classifica um ponto crítico via eigenvalues da Hessiana."""
    eigs = np.linalg.eigvalsh(hess)
    # TODO: se todos eigs > tol, devolve 'min'
    # TODO: se todos eigs < -tol, devolve 'max'
    # TODO: caso contrário, devolve 'saddle'
    tol = 1e-9
    return ...

# Sela em (0,0)
H_saddle = saddle_hess(np.array([0.0, 0.0]))
check(classify_critical_point(H_saddle) == 'saddle',
      f"f(x,y)=x²-y² em (0,0) deve ser sela, obteve {classify_critical_point(H_saddle)}")

# Mínimo em (0,0) de f = x² + y²
H_min = np.array([[2.0, 0.0], [0.0, 2.0]])
check(classify_critical_point(H_min) == 'min', 'x²+y² em (0,0) é mínimo')

# Máximo em (0,0) de f = -x² - y²
H_max = np.array([[-2.0, 0.0], [0.0, -2.0]])
check(classify_critical_point(H_max) == 'max', '-x²-y² em (0,0) é máximo')

# Em Rosenbrock, no mínimo (1, 1), Hessiana deve ser definida positiva
H_at_min = rosenbrock_hess(np.array([1.0, 1.0]))
eigs_at_min = np.linalg.eigvalsh(H_at_min)
print(f'eigenvalues de H_rosenbrock em (1,1): {eigs_at_min}')
check(classify_critical_point(H_at_min) == 'min', f'(1,1) em Rosenbrock é mínimo')
check(np.all(eigs_at_min > 0), 'eigenvalues > 0 confirmam mínimo')

## 🏁 Sprint 1 — fechado

Se chegaste aqui sem `AssertionError`, dominaste:

1. **Vetores e GD 1D** (CP01) — sigmoid, dot, norma, ângulo, GD básico, sensibilidade ao `lr`.
2. **Matrizes e convexidade** (CP02) — transposta, multiplicação, MSE, conjuntos convexos.
3. **Aceleração** (CP03) — Momentum, Nesterov, AdaGrad em vales alongados e regressão linear.
4. **Newton e adaptativos** (CP04) — gradiente/Hessiana numéricos, damped Newton com line search, RMSProp, Adam, AdamW.
5. **Showdown e paisagem** (CP05, este) — gradient check, Armijo, Nadam, scheduler cosine, comparação dos 7 otimizadores em Rosenbrock, classificação de pontos críticos.

**Perguntas finais (responde sem ver):**

- Em que tipo de problema é que **GD vanilla bate Adam**? (Pista: existe — pensa em quadráticas bem-condicionadas.)
- Qual o argumento intuitivo para o **número de iterações ser linear no condition number** $\kappa = \lambda_{\max}/\lambda_{\min}$ em GD?
- Se descobrires um ponto onde $\nabla f = 0$ mas a Hessiana tem um eigenvalue zero, o que isso significa? Como decides se é mínimo, máximo ou outra coisa?

**Próximo sprint:** Sprint 2 (Phase 1 final) — regressão linear/logística com regularização (Ridge, Lasso) + uma rede neural pequena from scratch. A álgebra que construíste aqui é o combustível.

**Entrega opcional:** escreve um relatório de 1 página (LaTeX ou Markdown) em `solutions/relatorio.md` com a tabela final do showdown e duas conclusões empíricas tuas.